# Customer Churn Prediction: Phase 3: Preprocessing & Feature Engineering


## 0. Setup


In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/telco_churn_clean.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

Loaded: 7,043 rows × 21 columns
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## Step 1: Drop `customerID`


In [2]:
df.drop(columns=['customerID'], inplace=True)
print(f'Shape after dropping customerID: {df.shape}')

Shape after dropping customerID: (7043, 20)


## Step 2: Audit All Categorical Columns


In [3]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns ({len(cat_cols)} total):\n')
print(f'{"Column":<25} {"Unique Values":<8} {"Values"}')
for col in cat_cols:
    vals = df[col].unique()
    print(f'{col:<25} {len(vals):<8} {list(vals)}')

Categorical columns (16 total):

Column                    Unique Values Values
gender                    2        ['Female', 'Male']
Partner                   2        ['Yes', 'No']
Dependents                2        ['No', 'Yes']
PhoneService              2        ['No', 'Yes']
MultipleLines             3        ['No phone service', 'No', 'Yes']
InternetService           3        ['DSL', 'Fiber optic', 'No']
OnlineSecurity            3        ['No', 'Yes', 'No internet service']
OnlineBackup              3        ['Yes', 'No', 'No internet service']
DeviceProtection          3        ['No', 'Yes', 'No internet service']
TechSupport               3        ['No', 'Yes', 'No internet service']
StreamingTV               3        ['No', 'Yes', 'No internet service']
StreamingMovies           3        ['No', 'Yes', 'No internet service']
Contract                  3        ['Month-to-month', 'One year', 'Two year']
PaperlessBilling          2        ['Yes', 'No']
PaymentMethod             4

## Step 3: Label Encoding for Binary Columns


In [4]:
# Binary Yes/No columns 
# Note: Several service columns use 'No internet service' or 'No phone service'
# as a third value. These are functionally equivalent to 'No' — the customer
# simply doesn't have that add-on. We consolidate them to 'No' first.

# Columns that have 'No internet service' as a variant of 'No'
internet_service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in internet_service_cols:
    df[col] = df[col].replace('No internet service', 'No')

# MultipleLines has 'No phone service' as a variant of 'No'
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

print('Replaced service variants. New unique values:')
for col in internet_service_cols + ['MultipleLines']:
    print(f'  {col}: {df[col].unique()}')

Replaced service variants. New unique values:
  OnlineSecurity: ['No' 'Yes']
  OnlineBackup: ['Yes' 'No']
  DeviceProtection: ['No' 'Yes']
  TechSupport: ['No' 'Yes']
  StreamingTV: ['No' 'Yes']
  StreamingMovies: ['No' 'Yes']
  MultipleLines: ['No' 'Yes']


In [5]:
# Now define all truly binary columns and map them
binary_map = {'Yes': 1, 'No': 0,
              'Male': 1, 'Female': 0}   # gender: arbitrary but consistent

binary_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

for col in binary_cols:
    df[col] = df[col].map(binary_map)

# Encode the TARGET separately
# WHY separately? So it doesn't accidentally get included in X.
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('Binary encoding done. Sample:')
df[binary_cols + ['Churn']].head()

Binary encoding done. Sample:


,gender,Partner,Dependents,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,Churn
0,0,1,0,0,0,0,1,0,0,0,0,1,0
1,1,0,0,1,0,1,0,1,0,0,0,0,0
2,1,0,0,1,0,1,1,0,0,0,0,1,1
3,1,0,0,0,0,1,0,1,1,0,0,0,0
4,0,0,0,1,0,0,0,0,0,0,0,1,1


## Step 4: One-Hot Encoding for Multi-Category Columns


In [6]:
# Columns with 3+ categories
ohe_cols = ['InternetService', 'Contract', 'PaymentMethod']

print('Unique values before encoding:')
for col in ohe_cols:
    print(f'  {col}: {df[col].unique()}')

# pd.get_dummies creates binary columns for each category
# drop_first=True drops the first dummy column to avoid multicollinearity
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print(f'\nShape after One-Hot Encoding: {df.shape}')
print('\nNew columns created:')
new_cols = [c for c in df.columns
            if any(c.startswith(p) for p in ['InternetService_', 'Contract_', 'PaymentMethod_'])]
print(new_cols)

Unique values before encoding:
  InternetService: ['DSL' 'Fiber optic' 'No']
  Contract: ['Month-to-month' 'One year' 'Two year']
  PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']

Shape after One-Hot Encoding: (7043, 24)

New columns created:
['InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## Step 5: Separate Features (X) and Target (y)


In [7]:
X = df.drop(columns=['Churn'])
y = df['Churn']

print(f'Feature matrix X: {X.shape}  ({X.shape[1]} features)')
print(f'Target vector  y: {y.shape}')
print(f'\nTarget distribution:')
print(f'  Class 0 (No Churn) : {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)')
print(f'  Class 1 (Churn)    : {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)')
print(f'\nFeature columns ({len(X.columns)}):')
print(list(X.columns))

Feature matrix X: (7043, 23)  (23 features)
Target vector  y: (7043,)

Target distribution:
  Class 0 (No Churn) : 5,174 (73.5%)
  Class 1 (Churn)    : 1,869 (26.5%)

Feature columns (23):
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


## Step 6: Feature Scaling with StandardScaler


In [8]:
# Identify which columns need scaling
# Only true continuous/count features — NOT binary dummies
cols_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges']

print('Columns that WILL be scaled (StandardScaler):')
print(f'  {cols_to_scale}')
print()
print('Current ranges (before scaling):')
print(X[cols_to_scale].describe().loc[['min', 'max', 'mean', 'std']].round(2))

Columns that WILL be scaled (StandardScaler):
  ['tenure', 'MonthlyCharges', 'TotalCharges']

Current ranges (before scaling):
      tenure  MonthlyCharges  TotalCharges
min     0.00           18.25          0.00
max    72.00          118.75       8684.80
mean   32.37           64.76       2279.73
std    24.56           30.09       2266.79


## Step 7: Final Data Quality Check


In [9]:
print('FINAL DATA QUALITY CHECK')
print(f'Total features       : {X.shape[1]}')
print(f'Total samples        : {X.shape[0]:,}')
print(f'Any nulls in X?      : {X.isnull().any().any()}')
print(f'Any nulls in y?      : {y.isnull().any()}')
print(f'All columns numeric?  : {all(X.dtypes != object)}')
print()

# Check all values are in expected range
binary_feature_cols = [c for c in X.columns if c not in cols_to_scale]
invalid_binary = [(c, X[c].unique()) for c in binary_feature_cols
                  if not set(X[c].unique()).issubset({0, 1, True, False})]
if invalid_binary:
    print('WARNING — unexpected values in binary columns:')
    for col, vals in invalid_binary:
        print(f'  {col}: {vals}')
else:
    print('All binary columns contain only 0/1 values.')

print()
print('Data types:')
print(X.dtypes.value_counts())

FINAL DATA QUALITY CHECK
Total features       : 23
Total samples        : 7,043
Any nulls in X?      : False
Any nulls in y?      : False
All columns numeric?  : True

All binary columns contain only 0/1 values.

Data types:
int64      14
bool        7
float64     2
Name: count, dtype: int64


## Step 8: Save Preprocessed Data


In [10]:
import pickle

# Save X and y as CSV
X.to_csv('../data/processed/X_preprocessed.csv', index=False)
y.to_csv('../data/processed/y_target.csv', index=False)

# Save cols_to_scale list — Phase 4 needs to know which columns to scale
with open('../data/processed/cols_to_scale.pkl', 'wb') as f:
    pickle.dump(cols_to_scale, f)

print('Saved:')
print('  data/processed/X_preprocessed.csv  (feature matrix)')
print('  data/processed/y_target.csv         (target vector)')
print('  data/processed/cols_to_scale.pkl    (column list for scaler)')
print()
print('X shape:', X.shape)
print('y shape:', y.shape)
print()


Saved:
  data/processed/X_preprocessed.csv  (feature matrix)
  data/processed/y_target.csv         (target vector)
  data/processed/cols_to_scale.pkl    (column list for scaler)

X shape: (7043, 23)
y shape: (7043,)

